In [17]:
!pip install gtfparse
!pip install polars=='0.16.17'

In [18]:
!pip install pyarrow

In [19]:
!pip install anndata==0.8.0

In [1]:
from samalg import SAM
from Bio import SeqIO
from gtfparse import read_gtf
import pandas as pd
import pandas
import pyarrow
import pickle
import scanpy as sc
import numpy as np

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#open scdata to see how many gene matches you have
dat = sc.read_h5ad('../../Testing_Raw_Dat_RNASEQ_Joined/tot_dat_RI_genome.h5ad')

In [3]:
dat.var_names

Index(['A1CF', 'A4GALT', 'AAAS', 'AACS', 'AADAT', 'AAGAB', 'AAK1', 'AAMDC',
       'AAMP', 'AANAT',
       ...
       'ZSWIM7', 'ZSWIM8', 'ZSWIM9', 'ZUP1', 'ZW10', 'ZWILCH', 'ZWINT', 'ZXDC',
       'ZYX', 'ZZZ3'],
      dtype='object', length=23867)

In [9]:
#RI created using gffread -w -F
#AC and DR created using kb ref -i -g -f1
input_file = open("../../cDNA_fasta/RI_gffread_F_raw_test_prateek.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    print(item)
    break

ID: XM_069745363.1
Name: XM_069745363.1
Description: XM_069745363.1 CDS=154-2739 db_xref=GenBank:XM_069745363.1;experiment="COORDINATES: polyA evidence [ECO:0006239]";gbkey=mRNA;gene=LOC138657619;model_evidence="Supporting evidence includes similarity to: 6 Proteins, 2 long SRA reads";product="zinc finger protein 271-like, transcript variant X2";transcript_biotype=mRNA;CDS_db_xref=GenBank:XP_069601464.1;CDS_gbkey=CDS;CDS_product="zinc finger protein 271-like";protein_id=XP_069601464.1
Number of features: 0
Seq('cgtcaccacaggtccttccagTGCAGCCGCTCCGTCCTggctgtgtgcagctcg...cca', SingleLetterAlphabet())


In [10]:
#pull longest gene
input_file = open("../../cDNA_fasta/RI_gffread_F_raw_test_prateek.fa")
gene_dict = {}
for item in SeqIO.parse(input_file, "fasta"):
    #manually change gene_ID to match which field you would like to see in your BLAST table
    #for RI use 'gene=' and ';'
    #for AC and DR use 'gene:' and ' '
    gene_ID = item.description.split('gene=')[1].split(';')[0]
    if gene_ID in gene_dict.keys():
        if len(item.seq) > len(gene_dict[gene_ID].seq):
            gene_dict[gene_ID] = item
    else:
        gene_dict[gene_ID] = item
len(gene_dict)

32095

In [11]:
len(set(dat.var_names) & set(gene_dict.keys()))

23801

In [12]:
for item in gene_dict.keys():
    gene_dict[item].id = item
    gene_dict[item].name = item

with open("../../BLASTMAPPING/RI_ncbi_genome_curated_08292026_gffreadF.fa", "w") as handle:
    SeqIO.write(gene_dict.values(), handle, "fasta") 